In [27]:
from nemo_microservices.data_designer.essentials import (
    DataDesignerConfigBuilder,
    InferenceParameters,
    ModelConfig,
    NeMoDataDesignerClient,
    SamplerType,
    CategorySamplerParams,
    UUIDSamplerParams,
    DatetimeSamplerParams,
    UniformSamplerParams,
    LLMJudgeColumnConfig,
    SamplerColumnConfig,
    ExpressionColumnConfig,
    InfoType,
    Score
)

In [28]:
import os

base_url = os.environ['NEMO_DATADESIGNER_BASE_URL']

model_provider = os.environ['MODEL_PROVIDER']
system_prompt = os.environ['SYSTEM_PROMPT']

structure_model_id = os.environ['STRUCTURE_MODEL_ID']
structure_model_alias = os.environ['STRUCTURE_MODEL_ALIAS']

judge_model_id = os.environ['JUDGE_MODEL_ID']
judge_model_alias = os.environ['JUDGE_MODEL_ALIAS']

In [29]:
print(
    f"""base_url: {base_url}
model_provider: {model_provider}
system_prompt: {system_prompt}

structure_model_id: {structure_model_id}
structure_model_alias: {structure_model_alias}

judge_model_id: {judge_model_id}
judge_model_alias: {judge_model_alias}""")

base_url: http://localhost:8000/
model_provider: nvidiabuild
system_prompt: /no_think

structure_model_id: nvidia/nemotron-3-nano-30b-a3b
structure_model_alias: structure-model

judge_model_id: nvidia/nvidia-nemotron-nano-9b-v2
judge_model_alias: judge-model


In [30]:
data_designer_client = NeMoDataDesignerClient(base_url=base_url, timeout=7200)

In [31]:
model_configs = [
  ModelConfig(
    provider=model_provider,
    model=structure_model_id,
    alias=structure_model_alias,
    inference_parameters=InferenceParameters(
      temperature=0.7,
      top_p=1.0,
      max_tokens=2048,
    ),
  ),
  ModelConfig(
    provider=model_provider,
    model=judge_model_id,
    alias=judge_model_alias,
    inference_parameters=InferenceParameters(
      temperature=0.3,
      top_p=0.5,
      max_tokens=1024,
    ),
  )
]

In [32]:
config_builder = DataDesignerConfigBuilder(model_configs=model_configs)

In [33]:
config_builder.info.display(InfoType.SAMPLERS)

─────────────────────────────────────────── NeMo Data Designer Samplers ───────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Type               ┃ Parameter                ┃ Data Type                         ┃ Required ┃ Constraints      ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ bernoulli          │ p                        │ number                            │    ✓     │ >= 0.0, <= 1.0   │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ bernoulli_mixture  │ p                        │ number                            │    ✓     │ >= 0.0, <= 1.0   │
│                    │ dist_name                │ string                            │    ✓     │                  │
│                    │ dist_params              │ dict                              │    ✓     │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ binomial           │ n                        │ integer                           │    ✓     │                  │
│                    │ p                        │ number                            │    ✓     │ >= 0.0, <= 1.0   │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ category           │ values                   │ string[] | integer[] | number[]   │    ✓     │ len > 1          │
│                    │ weights                  │ number[] | null                   │          │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ datetime           │ start                    │ string                            │    ✓     │                  │
│                    │ end                      │ string                            │    ✓     │                  │
│                    │ unit                     │ string                            │          │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ gaussian           │ mean                     │ number                            │    ✓     │                  │
│                    │ stddev                   │ number                            │    ✓     │                  │
│                    │ decimal_places           │ integer | null                    │          │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ person             │ locale                   │ string                            │          │                  │
│                    │ sex                      │ string | null                     │          │                  │
│                    │ city                     │ string | string[] | null          │          │                  │
│                    │ age_range                │ integer[]                         │          │ len > 2, len < 2 │
│                    │ select_field_values      │ objec

In [34]:
from pydantic import BaseModel, Field

class Transaction(BaseModel):
  total_quantity_7d: float = Field(description="Total Quantity of purchased items in the last 7 days")
  reported_transaction_total: int = Field(description="Reported Transaction Total")

In [ ]:
config_builder.add_column(
    SamplerColumnConfig(
        name="timestamp",
        sampler_type=SamplerType.DATETIME,
        params=DatetimeSamplerParams(start="2021-1-1", end="2026-1-1"),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="transaction_id",
        sampler_type=SamplerType.UUID,
        params=UUIDSamplerParams(short_form=True),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="customer_id",
        sampler_type=SamplerType.UUID,
        params=UUIDSamplerParams(short_form=False),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="transaction_total",
        sampler_type=SamplerType.UNIFORM,
        params=UniformSamplerParams(low=1000, high=10_000_000, decimal_places=0),
        convert_to="int",
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="merchant_category",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=["Electronics", "Clothing", "Groceries", "Home & Garden", "Sports"],
            weights=[0.2, 0.3, 0.25, 0.15, 0.1],
        ),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="payment_method",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=["Credit Card", "Debit Card", "PayPal", "Bank Transfer"],
            weights=[0.3, 0.3, 0.10, 0.3],
        ),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="region",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=["Jakarta", "Surabaya", "Bandung", "Medan", "Semarang"],
            weights=[0.3, 0.25, 0.2, 0.15, 0.1],
        ),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="total_quantity",
        sampler_type=SamplerType.UNIFORM,
        params=UniformSamplerParams(low=1, high=100, decimal_places=0),
        convert_to="int",
    )
)

# Establish the anomaly types (90% normal, 10% split between the 3 fraud types)
config_builder.add_column(
    SamplerColumnConfig(
        name="anomaly_type",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=["None", "QuantityOnly", "TotalOnly", "Both"],
            weights=[0.90, 0.04, 0.04, 0.02], # Adjust these weights as you see fit!
        ),
    )
)

# Generate random ratio multipliers for both scenarios
config_builder.add_column(
    SamplerColumnConfig(
        name="high_ratio",
        sampler_type=SamplerType.UNIFORM,
        params=UniformSamplerParams(low=2.1, high=5.0, decimal_places=2),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="normal_ratio",
        sampler_type=SamplerType.UNIFORM,
        params=UniformSamplerParams(low=0.5, high=2.0, decimal_places=2),
    )
)

# Calculate total_quantity_7d based on the specific anomaly type
config_builder.add_column(
    ExpressionColumnConfig(
        name="total_quantity_7d",
        expr="{{ (total_quantity / high_ratio if anomaly_type in ['QuantityOnly', 'Both'] else total_quantity / normal_ratio) | round(2) }}",
        dtype="float",
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="reduction_factor",
        sampler_type=SamplerType.UNIFORM,
        params=UniformSamplerParams(low=0.1, high=0.9, decimal_places=2),
    )
)

config_builder.add_column(
    ExpressionColumnConfig(
        name="reported_transaction_total",
        expr="{{ (transaction_total * reduction_factor)|int if anomaly_type in ['TotalOnly', 'Both'] else transaction_total }}",
        dtype="int",
    )
)

config_builder.add_column(
    LLMJudgeColumnConfig(
        name="fraud_analysis",
        model_alias=judge_model_alias,
        prompt=(
            "Determine if the transaction is fraudulent based on the following conditions:\n"
            "1. If the {{reported_transaction_total - transaction_total}} is not equal to 0, it indicates a discrepancy between the expected and reported transaction totals, which is a strong indicator of potential fraud.\n"
            "2. If the {{(total_quantity / total_quantity_7d) | round(2)}} is greater than or equal to 2.0, it suggests a significant increase in quantity compared to the average, which could also indicate fraudulent activity.\n"
            "If either of these conditions is met, classify the transaction as fraudulent (True). Otherwise, classify it as non-fraudulent (False)."
        ),
        scores=[
            Score(
                name="is_fraud",
                description="Indicates whether the transaction is fraudulent (1) or not (0).",
                options={
                    0: "The transaction is not fraud",
                    1: "The transaction is fraud",
                },
            )
        ],
    )
)

config_builder.validate()

[13:58:54] [INFO] ✅ Validation passed


DataDesignerConfigBuilder(
    sampler_columns: [
        "timestamp",
        "transaction_id",
        "customer_id",
        "transaction_total",
        "merchant_category",
        "payment_method",
        "region",
        "total_quantity",
        "anomaly_type",
        "high_ratio",
        "normal_ratio",
        "reduction_factor"
    ]
    llm_judge_columns: ['fraud_analysis']
    expression_columns: ['total_quantity_7d', 'reported_transaction_total']
)

In [36]:
preview = data_designer_client.preview(config_builder, num_records=10)

[13:58:54] [INFO] ✅ Validation passed
[13:58:54] [INFO] 🚀 Starting preview generation
[13:58:54] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[13:58:54] [INFO] 🩺 Running health checks for models...
[13:58:56] [INFO]   |-- 👀 Checking 'nvidia/nvidia-nemotron-nano-9b-v2' in provider named 'nvidiabuild' for model alias 'judge-model'...
[13:58:56] [INFO]   |-- ✅ Passed!
[13:58:56] [INFO] ⏳ Processing batch 1 of 1
[13:58:56] [INFO] 🎲 Preparing samplers to generate 10 records across 12 columns
[13:58:56] [INFO] 🧩 Generating column `total_quantity_7d` from expression
[13:58:57] [INFO] 🧩 Generating column `reported_transaction_total` from expression
[13:58:57] [INFO] ⚖️ Preparing llm-judge column generation
[13:58:57] [INFO]   |-- column name: 'fraud_analysis'
[13:58:57] [INFO]   |-- model config:
{
    "alias": "judge-model",
    "model": "nvidia/nvidia-nemotron-nano-9b-v2",
    "inference_parameters": {
        "temperature": 0.3,
        "top_p": 0.5,
        "max_tokens": 

In [37]:
preview.display_sample_record()

                                                 Generated Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                                              ┃ Value                                                       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ timestamp                                         │ 2021-10-04                                                  │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ transaction_id                                    │ d20fcf1a                                                    │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ customer_id                                       │ 71c9ef5a321d4de6b80d792fe0a0b270                            │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ transaction_total                                 │ 897534                                                      │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ merchant_category                                 │ Clothing                                                    │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ payment_method                                    │ Debit Card                                                  │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ region                                            │ Medan                                                       │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ total_quantity                                    │ 2                                                           │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ anomaly_type                                      │ None                                                        │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ high_ratio                                        │ 3.42                                                        │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ normal_ratio                                      │ 1.09                                                        │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ reduction_factor                                  │ 0.27                                                        │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ total_quantity_7d                                 │ 1.83                                                        │
├───────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────┤
│ reported_transaction_total                        │ 897534                                                      │
└───────────────────────────────────────────────────┴─────────────────────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                          LLM-as-a-Judge

In [38]:
preview.dataset

,timestamp,transaction_id,customer_id,transaction_total,merchant_category,payment_method,region,total_quantity,anomaly_type,high_ratio,normal_ratio,reduction_factor,total_quantity_7d,reported_transaction_total,fraud_analysis,fraud_analysis__reasoning_trace
0,2021-10-04,d20fcf1a,71c9ef5a321d4de6b80d792fe0a0b270,897534,Clothing,Debit Card,Medan,2,None,3.42,1.09,0.27,1.83,897534,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
1,2021-01-23,1af7d5d2,8d992fd86d9c4a888d2fadcdb245ce99,5978957,Sports,Credit Card,Medan,21,None,2.54,1.18,0.79,17.80,5978957,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
2,2025-10-11,a07c1c76,754dc06043e54ab48004af0057aa73f8,3330868,Electronics,Debit Card,Surabaya,57,None,3.34,0.56,0.11,101.79,3330868,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
3,2025-02-22,e11ef0e4,f156d7a84768410a833d2d664dc4c925,1084771,Groceries,Bank Transfer,Surabaya,50,TotalOnly,3.80,1.92,0.89,26.04,965446,{'is_fraud': {'reasoning': 'The transaction ha...,"Okay, let's see. I need to determine if the tr..."
4,2024-11-06,2ab8c89b,8db493a77e674193bd52e82cc8acc513,2615567,Groceries,Debit Card,Surabaya,16,None,3.40,1.33,0.23,12.03,2615567,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
5,2023-09-29,de606c44,a6d90dd1d4714260b33b536158a7d36f,8712404,Groceries,PayPal,Surabaya,13,None,4.44,0.78,0.10,16.67,8712404,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
6,2024-04-13,29113feb,a4df6007188d4c8b9e748ea5920750e8,1321038,Electronics,Debit Card,Bandung,14,None,4.64,1.02,0.36,13.73,1321038,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
7,2023-10-01,da5e011d,01e46e13b79b445096df6e42935e836d,9259437,Electronics,Credit Card,Jakarta,93,None,4.46,0.51,0.76,182.35,9259437,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
8,2021-09-19,44dc6d9a,2ce87224427e444b8e7035254a9c4715,9203011,Groceries,Debit Card,Surabaya,44,Both,2.19,1.44,0.73,20.09,6718198,{'is_fraud': {'reasoning': 'Both conditions ar...,"Okay, let's tackle this problem. The user want..."
9,2022-01-20,c0b010e0,2ff8825e75064f7a815fe3878a4b9d91,3162621,Home & Garden,Bank Transfer,Semarang,19,None,2.34,1.17,0.65,16.24,3162621,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."


In [39]:
if preview.analysis is not None:
	preview.analysis.to_report(save_path="analysis_report.html")

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 15                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃         data type ┃                number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ timestamp                      │            string │                         10 (100.0%) │             datetime │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ transaction_id                 │            string │                         10 (100.0%) │                 uuid │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ customer_id                    │            string │                         10 (100.0%) │                 uuid │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ transaction_total              │               int │                         10 (100.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ merchant_category              │            string │                           5 (50.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ payment_method                 │            string │                           4 (40.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ region                         │            string │                           5 (50.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ total_quantity                 │               int │                         10 (100.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ anomaly_type                   │            string │                           3 (30.0%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ high_ratio                     │             float │                         10 (100.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ normal_ratio                   │             float │   

In [40]:
job_results = data_designer_client.create(config_builder, num_records=1000)
job_results.wait_until_done()

[13:59:14] [INFO] 🎨 Creating Data Designer generation job
[13:59:14] [INFO] ✅ Validation passed
[13:59:14] [INFO]   |-- job_id: job-4hoesuvjzufeymnv4j71ln
[14:28:26] [INFO] 🥳 Dataset generation completed successfully.


In [41]:
df = job_results.load_dataset()
df.head()

,timestamp,transaction_id,customer_id,transaction_total,merchant_category,payment_method,region,total_quantity,anomaly_type,high_ratio,normal_ratio,reduction_factor,total_quantity_7d,reported_transaction_total,fraud_analysis,fraud_analysis__reasoning_trace
0,2024-09-14,661dded2,76f6e63b3e664d3daefdf63e4f511646,6643972,Clothing,Credit Card,Surabaya,46,None,2.75,0.92,0.58,50.0,6643972,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
1,2024-12-30,08a19d6a,70153d9ed06641c688cf3f18d15d0b78,5348844,Groceries,Debit Card,Bandung,43,None,3.18,1.81,0.88,23.76,5348844,{'is_fraud': {'reasoning': 'The first conditio...,"Okay, let's see. I need to determine if the tr..."
2,2025-01-28,0e7b5c27,83f7d3ae3e6c4539a48acd59b1d4ae5e,1523903,Groceries,Bank Transfer,Surabaya,4,None,3.98,0.96,0.8,4.17,1523903,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
3,2025-09-22,aba521c0,0ecfff90aae346999281c584944b241e,8429969,Clothing,Credit Card,Bandung,40,None,4.31,1.04,0.61,38.46,8429969,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."
4,2021-06-08,7a022f30,b5bc28295cee4ed7abe7efee8dce89f9,2370554,Clothing,Bank Transfer,Bandung,36,None,3.59,1.06,0.76,33.96,2370554,{'is_fraud': {'reasoning': 'Neither condition ...,"Okay, let's see. I need to determine if a tran..."


In [42]:
import json

fraud_analysis = df["fraud_analysis"].to_json()
fraud_analysis_json = json.loads(fraud_analysis)
print(fraud_analysis_json["0"])

{'is_fraud': {'reasoning': 'Neither condition was met. 0 equals 0 (no discrepancy), and 0.92 is less than 2.0 (no significant quantity increase).', 'score': '0'}}


In [43]:
def extract_fraud_analysis(fraud_analysis_json):
    fraud_analysis_list = []
    for key in fraud_analysis_json:
        fraud_analysis_list.append(
            {
                "is_fraud": fraud_analysis_json[key]["is_fraud"]["score"],
                "reasoning": fraud_analysis_json[key]["is_fraud"]["reasoning"],
            }
        )
    return fraud_analysis_list


fraud_analysis_list = extract_fraud_analysis(fraud_analysis_json)
df["is_fraud"] = [analysis["is_fraud"] for analysis in fraud_analysis_list]
df["reasoning"] = [analysis["reasoning"] for analysis in fraud_analysis_list]

In [44]:
df_cleaned = df.drop(
    columns=["fraud_analysis"],
    inplace=False,
)
df_cleaned["is_fraud"].value_counts()

is_fraud
0    896
1    104
Name: count, dtype: int64

In [45]:
%pip install plotly plotly[express]
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [51]:
import pandas as pd
df=pd.read_csv('synthetic_dataset_output/synthetic_transactions.csv')

In [46]:
import plotly.express as px
from plotly.colors import qualitative

fraud_mapping = {0: "Not Fraud", 1: "Fraud"}
distribution = df['is_fraud'].value_counts()
distribution.index = df["is_fraud"].value_counts().index.map(fraud_mapping)
distribution
# fig = px.bar(
#     distribution, 
#     title="Distribution of Fraudulent vs Non-Fraudulent Transactions",
#     x=distribution.index, 
#     y=distribution.values,
#     hover_name=distribution.index,
#     labels={"x": "Fraud Status", "y": "Count"}, 
#     color=distribution.index,
#     color_discrete_sequence=qualitative.T10,
#     width=600,
#     height=400,
#     custom_data=['']
# )
# fig.show()

is_fraud
Not Fraud    896
Fraud        104
Name: count, dtype: int64

In [47]:
# Load the analysis results into memory.
analysis = job_results.load_analysis()
analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1,000                           │ 15                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃         data type ┃                number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ timestamp                      │            string │                         776 (77.6%) │             datetime │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ transaction_id                 │            string │                       1000 (100.0%) │                 uuid │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ customer_id                    │            string │                       1000 (100.0%) │                 uuid │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ transaction_total              │               int │                       1000 (100.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ merchant_category              │            string │                            5 (0.5%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ payment_method                 │            string │                            4 (0.4%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ region                         │            string │                            5 (0.5%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ total_quantity                 │               int │                         100 (10.0%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ anomaly_type                   │            string │                            4 (0.4%) │             category │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ high_ratio                     │             float │                         283 (28.3%) │              uniform │
├────────────────────────────────┼───────────────────┼─────────────────────────────────────┼──────────────────────┤
│ normal_ratio                   │             float │   

In [48]:
SYNTHETIC_DATASET_OUTPUT = "synthetic_dataset_output"
job_results.download_artifacts(
  output_path=SYNTHETIC_DATASET_OUTPUT,
  artifacts_folder_name="synthetic_fraud_artifacts",
)

[14:28:33] [INFO] 🏺 Downloading artifacts from Job with ID 'job-4hoesuvjzufeymnv4j71ln'
[14:28:34] [INFO] ✅ Artifacts downloaded to synthetic_dataset_output\synthetic_fraud_artifacts


WindowsPath('synthetic_dataset_output/synthetic_fraud_artifacts')

In [49]:
columns_to_drop = [
    "anomaly_type",
    "high_ratio",
    "normal_ratio",
    "fraud_analysis",
    "fraud_analysis__reasoning_trace",
    "reduction_factor"
]
df = df.drop(columns=columns_to_drop)
df.to_csv(f"{SYNTHETIC_DATASET_OUTPUT}/synthetic_transactions.csv", index=False)